# Polymarket Data Exploration

Explore Polymarket prediction markets using the live API.

**No database setup required** - all data comes directly from Polymarket's public API.

See `API_GUIDE.md` in this folder for full documentation.

---

## 1. Setup

In [ ]:
from cuic_quant.notebook import pm
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 150)
pd.set_option('display.max_rows', 20)

print("Ready! Use pm.fetch_markets() and pm.fetch_orderbook() to get live data.")

## 2. Fetch Live Markets

Get real-time market data directly from Polymarket.

In [ ]:
# Fetch active markets from Polymarket API
df = pm.fetch_markets(limit=50, active=True)

print(f"Fetched {len(df)} markets from Polymarket API")
print()
df[['question', 'yes_price', 'volume', 'status']].head(15)

## 3. Analyze Markets

Basic analysis of the fetched data.

In [ ]:
# Sort by volume to find most active markets
top_markets = df.sort_values('volume', ascending=False)

print("TOP MARKETS BY VOLUME")
print("=" * 80)
for i, row in top_markets.head(10).iterrows():
    print(f"${row['volume']:>12,.0f} | {row['yes_price']:.0%} YES | {row['question'][:50]}...")

In [ ]:
# Price distribution
fig, ax = plt.subplots(figsize=(10, 5))
df['yes_price'].hist(bins=20, ax=ax, edgecolor='black', alpha=0.7)
ax.axvline(0.5, color='red', linestyle='--', label='50%')
ax.set_xlabel('Yes Price (Probability)')
ax.set_ylabel('Count')
ax.set_title('Market Price Distribution')
ax.legend()
plt.show()

## 4. Order Book Data

Fetch the bid/ask order book for a specific market.

Order books show the liquidity available at different price levels.

In [ ]:
import requests
import json

# Fetch open markets with token IDs
response = requests.get(
    "https://gamma-api.polymarket.com/markets", 
    params={"limit": 30, "closed": "false", "active": "true"}
)
raw_markets = response.json()

# Find a market with an orderbook
for market in raw_markets:
    clob_tokens = market.get("clobTokenIds")
    if not clob_tokens:
        continue
    
    if isinstance(clob_tokens, str):
        try:
            clob_tokens = json.loads(clob_tokens)
        except:
            continue
    
    if not clob_tokens:
        continue
    
    token_id = clob_tokens[0]  # YES token
    question = market.get("question", "Unknown")
    
    try:
        orderbook = pm.fetch_orderbook(token_id)
        
        if len(orderbook) > 0:
            print(f"Market: {question[:70]}...")
            print()
            
            bids = orderbook[orderbook['side'] == 'bid'].sort_values('price', ascending=False)
            asks = orderbook[orderbook['side'] == 'ask'].sort_values('price', ascending=True)
            
            print(f"BIDS ({len(bids)} levels)")
            if len(bids) > 0:
                print(bids[['price', 'size']].head(5).to_string())
            
            print(f"\nASKS ({len(asks)} levels)")
            if len(asks) > 0:
                print(asks[['price', 'size']].head(5).to_string())
            
            if len(bids) > 0 and len(asks) > 0:
                spread = asks['price'].min() - bids['price'].max()
                print(f"\nSpread: {spread:.4f} ({spread*100:.2f}%)")
            break
    except:
        continue
else:
    print("No markets with active orderbooks found")

## 5. Sports Markets (NBA)

Sports markets are organized under the `/events` endpoint.

In [ ]:
# Fetch NBA events
response = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"limit": 100, "closed": "false"}
)
events = response.json()

# Filter for NBA
nba_events = [e for e in events if "nba" in e.get("slug", "").lower()]

print(f"Found {len(nba_events)} NBA events")
print()

for event in nba_events[:5]:
    title = event.get("title", "N/A")
    markets = event.get("markets", [])
    total_volume = sum(float(m.get("volume", 0)) for m in markets)
    
    print(f"{title}")
    print(f"  Markets: {len(markets)} | Volume: ${total_volume:,.0f}")
    
    # Show top market by volume
    if markets:
        top = max(markets, key=lambda x: float(x.get("volume", 0)))
        print(f"  Top: {top.get('question', 'N/A')[:60]}...")
    print()

## 6. Direct API Access

For more control, use the Polymarket API directly.

In [ ]:
# Gamma API - Markets endpoint
response = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={
        "limit": 10,
        "active": "true",
        "closed": "false"
    }
)

markets = response.json()
print(f"Raw API response: {len(markets)} markets")
print()

# Show available fields
if markets:
    print("Available fields:")
    for key in sorted(markets[0].keys()):
        print(f"  - {key}")

---

## Summary

| Method | Description |
|--------|-------------|
| `pm.fetch_markets(limit, active)` | Get live markets from API |
| `pm.fetch_orderbook(token_id)` | Get bid/ask order book |
| Direct `requests.get()` | Full API access |

### API Endpoints

- **Markets:** `https://gamma-api.polymarket.com/markets`
- **Events:** `https://gamma-api.polymarket.com/events`
- **Orderbook:** `https://clob.polymarket.com/book?token_id=...`

See `API_GUIDE.md` for more details.